# Spoken digits: a guided audio study

Listen to a recording, inspect its features, and compare classifiers on a speaker excluded from training. Code and initial results were prepared with AI assistance. Add your own listening observations after running the notebook.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Audio, Image, display
from sklearn.metrics import confusion_matrix
from study import DATA, OUT, read_audio, features


## Listen to one recording
The label in the filename is the spoken digit; the next component identifies the speaker.

In [ ]:
sample = DATA / "recordings/7_jackson_0.wav"
rate, waveform = read_audio(sample)
print({"sample_rate": rate, "samples": len(waveform), "duration_seconds": len(waveform)/rate})
display(Audio(waveform, rate=rate))
display(Image(filename=str(OUT / "audio-example.png")))

## Inspect pooled and time-resolved features
The baseline reduces a clip to 79 statistics. The follow-up keeps 13 coefficients at 32 ordered time points, then normalises coefficients within each frame.

In [ ]:
from compare import representations
from normalized import normalise_frames
vector = features(waveform, rate)
temporal, log_mel = representations(waveform)
normalised = normalise_frames(temporal[None]).reshape(13, 32)
print("Pooled feature count:", len(vector))
print("Time-resolved feature count:", len(temporal))
print("Per-frame coefficient means (first five):", np.round(normalised.mean(axis=0)[:5], 5))
print("Per-frame coefficient stds (first five):", np.round(normalised.std(axis=0)[:5], 5))

## Compare evaluation questions and methods
The recording split includes familiar speakers. Speaker-held-out evaluation excludes the test voice from training. All models train on clean audio. The follow-up is stronger on clean audio and weaker with synthetic noise.

In [ ]:
metrics = json.loads((OUT / "metrics.json").read_text())
comparison = json.loads((OUT / "comparison.json").read_text())
normalised_metrics = json.loads((OUT / "normalized.json").read_text())
for label, report in [("Pooled baseline", metrics), ("Normalised temporal MFCCs", normalised_metrics)]:
    print(label)
    print("  Known speakers:", report["recording_split_known_speakers"])
    print("  Unseen speakers:", report["leave_one_speaker_out"])
print("CNN unseen speakers, clean:", comparison["leave_one_speaker_out"]["clean_cnn"])
display(Image(filename=str(OUT / "final-comparison.png")))

## Investigate mistakes
Change the speaker below and switch `prediction_file` between `normalized-predictions.npz` and `predictions.npz`. Compare confused digits and listen to two errors. These observations do not identify the cause of a speaker difference.

In [ ]:
held_out_speaker = "nicolas"
prediction_file = "normalized-predictions.npz"
results = np.load(OUT / prediction_file)
selected = results["groups"] == held_out_speaker
matrix = confusion_matrix(results["labels"][selected], results["clean"][selected], labels=np.arange(10))
print("Rows: true digits; columns: predicted digits")
print(matrix)
errors = np.flatnonzero(selected & (results["labels"] != results["clean"]))
for index in errors[:2]:
    filename = str(results["filenames"][index])
    print(filename, "predicted", int(results["clean"][index]))
    sr, sound = read_audio(DATA / "recordings" / filename)
    display(Audio(sound, rate=sr))

## Your observation
Write two or three sentences in your own words here after listening. This section is intentionally unfinished.

## Reproduce the full study
From this directory run `python study.py --download`, `python compare.py`, `python normalized.py`, then `python checks.py`. Dataset attribution, method sources, development history and limitations are documented in README.md and SOURCE-REVIEW.md.